
# PreCNN Embedding Inference & Position Estimation

Use the trained supervised AveCSI PreCNN as a frozen feature extractor to localize new CSI windows without re-training.



## Notebook outline

1. Configure paths, feature options, and inference hyper-parameters.
2. Load helper utilities to normalize CSI windows exactly like training.
3. Load the frozen PreCNN backbone and build reference / query tensors.
4. Optionally refresh BatchNorm statistics on the new session.
5. Embed reference CSI windows and build a database keyed by physical coordinates.
6. Embed query CSI windows, perform soft RBF-weighted nearest-neighbour localization, and inspect per-point weights.
7. Compute localization error metrics (median, percentiles, etc.) and save detailed predictions.


In [ ]:

import os
import json
from glob import glob
from collections import Counter, defaultdict
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers

# Session setup ---------------------------------------------------------------
tf.keras.backend.clear_session()
for g in tf.config.experimental.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

print("TensorFlow:", tf.__version__)

# ---------------------------------------------------------------------------
# User configuration: update the paths below to match your environment.
# ---------------------------------------------------------------------------
MODEL_BACKBONE_PATH = "/home/tonyliao/WIFI_SENSING_LOCATION/runs_windows_supervised_avecsi/encoder_windows_avecsi.h5"

REFERENCE_METADATA_CSV = "/path/to/reference_metadata.csv"
REFERENCE_ROOT = "/path/to/reference_windows"

QUERY_METADATA_CSV = "/path/to/query_metadata.csv"
QUERY_ROOT = "/path/to/query_windows"

# Feature processing must match the training run that produced the backbone.
FEATURE_MODE = "amp+phase+sin_cos"  # options: 'amp', 'amp+phase', 'amp+sin_cos', 'amp+phase+sin_cos'
FORCE_LAYOUT = "TC"                 # MATLAB exporter default: time-major (T, C)
CROP_LEN = None                     # Set to int to center-crop windows before padding
PAD_TO = None                       # If None, inferred from longest window in metadata

# Inference / localization parameters
BATCH_SIZE = 256
RBF_GAMMA = 1.0                     # Soft RBF temperature; higher = sharper weighting
SOFT_TOP_K = 8                      # Limit neighbours contributing to the soft RBF weight (None = all)
REPORT_TOP_K = 5                    # How many reference points to log per query

REFRESH_BATCH_NORM = True           # Run a BN "calibration" pass on the new session
SAVE_PREDICTIONS_CSV = "precNN_session_predictions.csv"  # None to skip saving


In [ ]:

print("Backbone checkpoint:", MODEL_BACKBONE_PATH)
print("Reference metadata:", REFERENCE_METADATA_CSV)
print("Query metadata:    ", QUERY_METADATA_CSV)
print("Feature mode:       {}".format(FEATURE_MODE))
print("Force layout:       {}".format(FORCE_LAYOUT))
print("Crop length:        {}".format(CROP_LEN))
print("Pad to length:      {}".format(PAD_TO))
print("BN refresh:         {}".format(REFRESH_BATCH_NORM))


In [ ]:

SUPPORTED_FEATURE_MODES = {"amp", "amp+phase", "amp+sin_cos", "amp+phase+sin_cos"}
if FEATURE_MODE not in SUPPORTED_FEATURE_MODES:
    raise ValueError(f"Unsupported FEATURE_MODE={FEATURE_MODE}. Valid: {sorted(SUPPORTED_FEATURE_MODES)}")


def ensure_channels_first(arr: np.ndarray, force_layout: str = "TC") -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape {arr.shape}")
    if force_layout == "TC":
        return arr.T
    if force_layout == "CT":
        return arr
    raise ValueError(f"Unknown force_layout '{force_layout}'")


def related_window_path(path: str, stem: str) -> str:
    base_dir, fname = os.path.split(path)
    if not fname.startswith("X_window_"):
        raise ValueError(f"Unexpected window filename: {fname}")
    suffix = fname[len("X_window_"):]
    return os.path.join(base_dir, f"{stem}{suffix}")


def wants_raw_phase(mode: str) -> bool:
    return "phase" in mode and mode != "amp+sin_cos"


def wants_sin_cos(mode: str) -> bool:
    return "sin_cos" in mode


def expected_feature_names(mode: str) -> List[str]:
    names: List[str] = []
    if mode != "amp":
        if wants_raw_phase(mode):
            names.append("phase")
        if wants_sin_cos(mode):
            names.extend(["phase_sin", "phase_cos"])
    return names


def load_window_components(path: str, feature_mode: str = "amp", force_layout: str = "TC"):
    amp = ensure_channels_first(np.load(path), force_layout=force_layout)
    extras: List[np.ndarray] = []
    names: List[str] = []
    missing: set[str] = set()

    if feature_mode != "amp":
        if wants_raw_phase(feature_mode):
            phase_path = related_window_path(path, "Xphase_window_")
            if os.path.isfile(phase_path):
                phase = ensure_channels_first(np.load(phase_path), force_layout=force_layout)
                extras.append(phase.astype(np.float32))
                names.append("phase")
            else:
                missing.add("phase")
                extras.append(np.zeros_like(amp, dtype=np.float32))
                names.append("phase")

        if wants_sin_cos(feature_mode):
            sc_path = related_window_path(path, "XphaseSC_window_")
            if os.path.isfile(sc_path):
                sc = np.load(sc_path)
                if sc.ndim != 3 or sc.shape[-1] != 2:
                    raise ValueError(f"{sc_path} expected shape [T, S, 2], got {sc.shape}")
                sin_comp = ensure_channels_first(sc[:, :, 0], force_layout=force_layout)
                cos_comp = ensure_channels_first(sc[:, :, 1], force_layout=force_layout)
                extras.extend([sin_comp.astype(np.float32), cos_comp.astype(np.float32)])
                names.extend(["phase_sin", "phase_cos"])
            else:
                missing.update({"phase_sin", "phase_cos"})
                extras.extend([
                    np.zeros_like(amp, dtype=np.float32),
                    np.zeros_like(amp, dtype=np.float32),
                ])
                names.extend(["phase_sin", "phase_cos"])

    return amp.astype(np.float32), extras, names, missing


def center_crop(arr: np.ndarray, length: int) -> np.ndarray:
    if arr.shape[1] <= length:
        return arr
    start = max((arr.shape[1] - length) // 2, 0)
    end = start + length
    return arr[:, start:end]


def pad_to_length(arr: np.ndarray, length: int) -> np.ndarray:
    if arr.shape[1] == length:
        return arr
    if arr.shape[1] > length:
        return arr[:, :length]
    pad_width = ((0, 0), (0, length - arr.shape[1]))
    return np.pad(arr, pad_width, mode="constant")


def compute_T_stats(paths: Sequence[str], force_layout: str = "TC", crop_len: Optional[int] = None):
    T_list: List[int] = []
    C_first: Optional[int] = None
    for p in paths:
        if not os.path.isfile(p):
            continue
        X = np.load(p)
        if X.ndim != 2:
            continue
        if force_layout == "TC":
            C, T = X.shape[1], X.shape[0]
        else:
            C, T = X.shape
        if C_first is None:
            C_first = int(C)
        T_list.append(int(T if crop_len is None else crop_len))
    if len(T_list) == 0:
        return C_first, None, None
    T_max = max(T_list) if crop_len is None else crop_len
    T_min = min(T_list) if crop_len is None else crop_len
    return C_first, T_min, T_max


def compute_session_norm_stats(paths: Sequence[str], feature_mode: str = "amp", force_layout: str = "TC"):
    amp_sum: Optional[np.ndarray] = None
    amp_sumsq: Optional[np.ndarray] = None
    amp_count: int = 0

    extra_acc: Dict[str, Dict[str, Optional[np.ndarray]]] = defaultdict(lambda: {"sum": None, "sumsq": None, "count": 0})
    missing_counter: Counter = Counter()

    for path in paths:
        if not os.path.isfile(path):
            continue
        amp, extras, names, missing = load_window_components(path, feature_mode=feature_mode, force_layout=force_layout)
        amp_sum = amp.sum(axis=1, keepdims=True) if amp_sum is None else amp_sum + amp.sum(axis=1, keepdims=True)
        amp_sumsq = (amp**2).sum(axis=1, keepdims=True) if amp_sumsq is None else amp_sumsq + (amp**2).sum(axis=1, keepdims=True)
        amp_count += amp.shape[1]

        for name, arr in zip(names, extras):
            if name in missing:
                missing_counter[name] += 1
                continue
            stats = extra_acc[name]
            stats["sum"] = arr.sum(axis=1, keepdims=True) if stats["sum"] is None else stats["sum"] + arr.sum(axis=1, keepdims=True)
            stats["sumsq"] = (arr**2).sum(axis=1, keepdims=True) if stats["sumsq"] is None else stats["sumsq"] + (arr**2).sum(axis=1, keepdims=True)
            stats["count"] = int(stats["count"]) + arr.shape[1]

    if amp_sum is None or amp_count == 0:
        raise RuntimeError("Unable to compute normalization stats; no valid CSI windows were found.")

    mu = amp_sum / float(amp_count)
    var = amp_sumsq / float(amp_count) - mu**2
    sigma = np.sqrt(np.maximum(var, 1e-6)).astype(np.float32)
    mu = mu.astype(np.float32)

    extra_stats: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for name, stats in extra_acc.items():
        count = int(stats["count"])
        if count == 0 or stats["sum"] is None or stats["sumsq"] is None:
            continue
        mu_e = stats["sum"] / float(count)
        var_e = stats["sumsq"] / float(count) - mu_e**2
        sigma_e = np.sqrt(np.maximum(var_e, 1e-6)).astype(np.float32)
        extra_stats[name] = (mu_e.astype(np.float32), sigma_e)

    info = {
        "total_windows": len(paths),
        "missing_features": dict(missing_counter),
        "extra_stats_available": sorted(extra_stats.keys()),
    }
    return mu, sigma, extra_stats, info


def build_feature_tensor(paths: Sequence[str], mu: Optional[np.ndarray], sigma: Optional[np.ndarray],
                         feature_mode: str = "amp", extra_stats: Optional[Dict[str, Tuple[np.ndarray, np.ndarray]]] = None,
                         force_layout: str = "TC", crop_len: Optional[int] = None, pad_to: Optional[int] = None):
    extra_stats = extra_stats or {}
    arrays: List[np.ndarray] = []
    component_order: Optional[List[str]] = None
    missing_counter: Counter = Counter()

    if crop_len is None and pad_to is None:
        raise ValueError("pad_to must be provided when crop_len is None")

    for path in paths:
        if not os.path.isfile(path):
            continue
        amp, extras, names, missing = load_window_components(path, feature_mode=feature_mode, force_layout=force_layout)
        if component_order is None:
            component_order = ["amp"] + names
        if mu is not None and sigma is not None:
            amp = (amp - mu) / (sigma + 1e-6)

        parts: List[np.ndarray] = [amp.astype(np.float32)]
        for name, arr in zip(names, extras):
            arr = arr.astype(np.float32)
            if name in missing:
                missing_counter[name] += 1
                arr = np.zeros_like(arr)
            elif name in extra_stats:
                mu_e, sigma_e = extra_stats[name]
                arr = (arr - mu_e) / (sigma_e + 1e-6)
            else:
                missing_counter[f"stats_missing::{name}"] += 1
            parts.append(arr)

        if crop_len is not None:
            parts = [center_crop(part, crop_len) for part in parts]

        target_pad = pad_to if pad_to is not None else crop_len
        parts = [pad_to_length(part, target_pad) for part in parts]

        arrays.append(np.concatenate(parts, axis=0).astype(np.float32))

    if not arrays:
        raise RuntimeError("No tensors were built; check that the metadata points to existing windows.")

    X = np.stack(arrays, axis=0).astype(np.float32)
    info = {
        "component_order": component_order if component_order else ["amp"],
        "missing_features": dict(missing_counter),
        "total_windows": len(arrays),
        "pad_to": pad_to,
        "crop_len": crop_len,
    }
    return X, info


In [ ]:

def load_metadata(csv_path: str, root: Optional[str] = None) -> pd.DataFrame:
    if not csv_path:
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    required_cols = {"window_path", "point_id", "x", "y"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Metadata {csv_path} is missing required columns: {sorted(missing)}")
    if root:
        df["window_path"] = df["window_path"].apply(lambda p: os.path.join(root, p) if not os.path.isabs(p) else p)
    df["window_path"] = df["window_path"].astype(str)
    df = df.sort_values("window_path").reset_index(drop=True)
    missing_files = [p for p in df["window_path"] if not os.path.isfile(p)]
    if missing_files:
        print(f"[WARN] {len(missing_files)} metadata paths do not exist; they will be ignored in tensor builds.")
    return df


def refresh_batch_norm(model: keras.Model, data: np.ndarray, batch_size: int = 128) -> None:
    bn_layers = [layer for layer in model.layers if isinstance(layer, layers.BatchNormalization)]
    if not bn_layers:
        print("[BN] No BatchNormalization layers detected; skipping refresh.")
        return
    print(f"[BN] Refreshing running statistics on {data.shape[0]} samples...")
    ds = tf.data.Dataset.from_tensor_slices(data.astype(np.float32)).batch(batch_size)
    for batch in ds:
        model(batch, training=True)
    print("[BN] Refresh complete.")


def extract_embeddings(model: keras.Model, data: np.ndarray, batch_size: int = 256) -> np.ndarray:
    if data.size == 0:
        return np.empty((0, model.output_shape[-1]), dtype=np.float32)
    return model.predict(data, batch_size=batch_size, verbose=1)


def build_reference_database(meta: pd.DataFrame, embeddings: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    if len(meta) != len(embeddings):
        raise ValueError("Metadata and embedding counts do not match.")
    grouped = []
    for point_id, group in meta.groupby("point_id"):
        idx = group.index.to_numpy()
        mean_emb = embeddings[idx].mean(axis=0)
        coords = group[["x", "y"]].mean().to_numpy(dtype=np.float32)
        grouped.append({
            "point_id": point_id,
            "x": float(coords[0]),
            "y": float(coords[1]),
            "count": int(len(group)),
            "embedding": mean_emb.astype(np.float32),
        })
    ref_df = pd.DataFrame(grouped).sort_values("point_id").reset_index(drop=True)
    ref_matrix = np.stack(ref_df["embedding"].to_numpy(), axis=0)
    ref_coords = ref_df[["x", "y"]].to_numpy(dtype=np.float32)
    return ref_df, ref_matrix, ref_coords


def soft_rbf_localization(query_embeddings: np.ndarray, ref_embeddings: np.ndarray, ref_coords: np.ndarray,
                           gamma: float = 1.0, top_k: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray]:
    if query_embeddings.size == 0:
        return np.empty((0, ref_coords.shape[1]), dtype=np.float32), np.empty((0, ref_embeddings.shape[0]), dtype=np.float32)
    preds = []
    weight_records = []
    for q in query_embeddings:
        diff = ref_embeddings - q[None, :]
        dist2 = np.sum(diff * diff, axis=1)
        weights = np.exp(-gamma * dist2)
        if top_k is not None and 0 < top_k < len(weights):
            idx = np.argpartition(dist2, top_k)[:top_k]
            mask = np.zeros_like(weights)
            mask[idx] = 1.0
            weights *= mask
        total = weights.sum()
        if total <= 0:
            weights = np.ones_like(weights) / len(weights)
        else:
            weights /= total
        preds.append(weights @ ref_coords)
        weight_records.append(weights)
    return np.stack(preds, axis=0).astype(np.float32), np.stack(weight_records, axis=0).astype(np.float32)


def summarize_errors(errors: np.ndarray) -> Dict[str, float]:
    errors = np.asarray(errors, dtype=np.float32)
    finite = errors[np.isfinite(errors)]
    if finite.size == 0:
        return {"median": float("nan"), "mean": float("nan"), "p75": float("nan"), "p90": float("nan"), "max": float("nan")}
    return {
        "median": float(np.median(finite)),
        "mean": float(np.mean(finite)),
        "p75": float(np.percentile(finite, 75)),
        "p90": float(np.percentile(finite, 90)),
        "max": float(np.max(finite)),
    }


def format_reference_mix(weights: np.ndarray, reference_df: pd.DataFrame, top_k: int = 5) -> List[Dict[str, float]]:
    if weights.size == 0:
        return []
    order = np.argsort(weights)[::-1][:top_k]
    mix = []
    for idx in order:
        entry = reference_df.iloc[int(idx)]
        mix.append({
            "point_id": entry["point_id"],
            "weight": float(weights[idx]),
            "x": float(entry["x"]),
            "y": float(entry["y"]),
        })
    return mix


In [ ]:

reference_meta = load_metadata(REFERENCE_METADATA_CSV, REFERENCE_ROOT)
query_meta = load_metadata(QUERY_METADATA_CSV, QUERY_ROOT)

all_paths: List[str] = []
if not reference_meta.empty:
    all_paths.extend(reference_meta["window_path"].tolist())
if not query_meta.empty:
    all_paths.extend(query_meta["window_path"].tolist())
all_paths = sorted({p for p in all_paths if os.path.isfile(p)})

if not all_paths:
    raise RuntimeError("No valid CSI window files discovered across reference/query metadata.")

C_detected, T_min, T_max = compute_T_stats(all_paths, force_layout=FORCE_LAYOUT, crop_len=CROP_LEN)
print(f"Detected {len(all_paths)} windows with channel count {C_detected} and time range [{T_min}, {T_max}]")

pad_length = PAD_TO if PAD_TO is not None else T_max
print(f"Using pad length: {pad_length}")

mu_session, sigma_session, extra_stats_session, stats_info = compute_session_norm_stats(all_paths, feature_mode=FEATURE_MODE, force_layout=FORCE_LAYOUT)
print("Session mu shape:", mu_session.shape, "sigma shape:", sigma_session.shape)
print("Extra feature stats for:", stats_info.get("extra_stats_available", []))
if stats_info.get("missing_features"):
    print("[WARN] Missing feature signals encountered:", stats_info["missing_features"])


In [ ]:

backbone = keras.models.load_model(MODEL_BACKBONE_PATH)
backbone.trainable = False
print("Backbone output shape:", backbone.output_shape)


In [ ]:

reference_tensors, reference_info = (None, None)
if not reference_meta.empty:
    reference_tensors, reference_info = build_feature_tensor(
        reference_meta["window_path"].tolist(),
        mu_session, sigma_session,
        feature_mode=FEATURE_MODE,
        extra_stats=extra_stats_session,
        force_layout=FORCE_LAYOUT,
        crop_len=CROP_LEN,
        pad_to=pad_length,
    )
    print("Reference tensor shape:", reference_tensors.shape)
    print("Reference component order:", reference_info.get("component_order"))
    if reference_info.get("missing_features"):
        print("[WARN] Reference missing features:", reference_info["missing_features"])
else:
    reference_tensors = np.empty((0, 0, 0), dtype=np.float32)

query_tensors, query_info = (None, None)
if not query_meta.empty:
    query_tensors, query_info = build_feature_tensor(
        query_meta["window_path"].tolist(),
        mu_session, sigma_session,
        feature_mode=FEATURE_MODE,
        extra_stats=extra_stats_session,
        force_layout=FORCE_LAYOUT,
        crop_len=CROP_LEN,
        pad_to=pad_length,
    )
    print("Query tensor shape:", query_tensors.shape)
    print("Query component order:", query_info.get("component_order"))
    if query_info.get("missing_features"):
        print("[WARN] Query missing features:", query_info["missing_features"])
else:
    query_tensors = np.empty((0, 0, 0), dtype=np.float32)


In [ ]:

if REFRESH_BATCH_NORM:
    calibration_batches = []
    if reference_tensors.size:
        calibration_batches.append(reference_tensors)
    if query_tensors.size:
        calibration_batches.append(query_tensors)
    if calibration_batches:
        calibration_array = np.concatenate(calibration_batches, axis=0)
        refresh_batch_norm(backbone, calibration_array, batch_size=BATCH_SIZE)
    else:
        print("[BN] Nothing to refresh; skipping.")


In [ ]:

reference_embeddings = np.empty((0, backbone.output_shape[-1]), dtype=np.float32)
if reference_tensors.size:
    reference_embeddings = extract_embeddings(backbone, reference_tensors, batch_size=BATCH_SIZE)
    print("Reference embeddings:", reference_embeddings.shape)

query_embeddings = np.empty((0, backbone.output_shape[-1]), dtype=np.float32)
if query_tensors.size:
    query_embeddings = extract_embeddings(backbone, query_tensors, batch_size=BATCH_SIZE)
    print("Query embeddings:", query_embeddings.shape)


In [ ]:

if reference_embeddings.size == 0:
    raise RuntimeError("Reference embeddings are empty; localization requires at least one reference point.")

reference_meta = reference_meta.reset_index(drop=True)
if len(reference_meta) != len(reference_embeddings):
    raise RuntimeError("Reference metadata and embeddings mismatch after extraction.")

reference_database, reference_matrix, reference_coords = build_reference_database(reference_meta, reference_embeddings)
print(reference_database[["point_id", "x", "y", "count"]])
print("Reference matrix shape:", reference_matrix.shape)


In [ ]:

if query_embeddings.size == 0:
    raise RuntimeError("Query embeddings are empty; provide query metadata/windows to localize.")

pred_xy, weight_matrix = soft_rbf_localization(
    query_embeddings,
    reference_matrix,
    reference_coords,
    gamma=RBF_GAMMA,
    top_k=SOFT_TOP_K,
)

query_results = query_meta.copy().reset_index(drop=True)
query_results["pred_x"] = pred_xy[:, 0]
query_results["pred_y"] = pred_xy[:, 1]
true_xy = query_results[["x", "y"]].to_numpy(dtype=np.float32)
query_results["error_m"] = np.linalg.norm(pred_xy - true_xy, axis=1)
query_results["reference_mix"] = [json.dumps(format_reference_mix(weights, reference_database, top_k=REPORT_TOP_K)) for weights in weight_matrix]

error_metrics = summarize_errors(query_results["error_m"].to_numpy())
print("Localization error summary (meters):")
for k, v in error_metrics.items():
    print(f"  {k}: {v:.4f}")

query_results.head()


In [ ]:

if SAVE_PREDICTIONS_CSV:
    query_results.to_csv(SAVE_PREDICTIONS_CSV, index=False)
    print("Saved predictions to", os.path.abspath(SAVE_PREDICTIONS_CSV))
